# 2.2 Runtime Context (adaptado a lab local)

**Convención de este fork:** 🔸 ORIGINAL DEL CURSO (comentada, con el motivo) → 🟢 ADAPTADO LOCAL (la que corre con Ollama). Celdas sin marca = iguales al curso.

**Qué se ve aquí:** el *runtime context* son datos que se pasan al agente **en cada invocación** (`context=...`), por fuera de los mensajes. El modelo **no los ve**: solo las tools (o el middleware) pueden leerlos vía `ToolRuntime`.

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: falta cargar local_model.py (está en la raíz del repo).
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from dotenv import load_dotenv
#
# load_dotenv()

In [1]:
# 🟢 ADAPTADO LOCAL
# Motivo: el notebook está en notebooks/module-2/ y local_model.py en la raíz del repo.
import sys, pathlib
sys.path.insert(0, str(next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "local_model.py").exists())))

from dotenv import load_dotenv
load_dotenv()
from local_model import get_model

In [2]:
# Sin cambios
from dataclasses import dataclass

@dataclass
class ColourContext:
    favourite_colour: str = "blue"
    least_favourite_colour: str = "yellow"

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: gpt-5-nano es API de pago.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from langchain.agents import create_agent
#
# agent = create_agent(
#     model="gpt-5-nano",
#     context_schema=ColourContext  
# )

In [3]:
# 🟢 ADAPTADO LOCAL
# Motivo: "gpt-5-nano" → get_model("gemma4:latest"). Este agente NO tiene tools:
# el context existe en el runtime pero el modelo no lo ve → debería responder que no sabe tu color.
from langchain.agents import create_agent

agent = create_agent(
    model=get_model("gemma4:latest"),
    context_schema=ColourContext
)

In [4]:
# Sin cambios
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

In [5]:
# Sin cambios
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='89050639-73ad-4781-8604-8fa34d5004f8'),
              AIMessage(content="I do not know what your favourite colour is. As an AI, I don't have access to your personal information unless you tell it to me.", additional_kwargs={}, response_metadata={'model': 'gemma4:latest', 'created_at': '2026-10-05T00:21:18.0540973Z', 'done': True, 'done_reason': 'stop', 'total_duration': 9748120800, 'load_duration': 7040944500, 'prompt_eval_count': 15, 'prompt_eval_duration': 502135000, 'eval_count': 32, 'eval_duration': 2202821000, 'logprobs': None, 'model_name': 'gemma4:latest', 'model_provider': 'ollama'}, id='lc_run--01a1096f-8cc5-7a13-9e15-6c08265c97b2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 32, 'total_tokens': 47})]}


## Accessing Context

In [6]:
# Sin cambios
from langchain.tools import tool, ToolRuntime

@tool
def get_favourite_colour(runtime: ToolRuntime[ColourContext]) -> str:
    """Get the favourite colour of the user"""
    return runtime.context.favourite_colour

@tool
def get_least_favourite_colour(runtime: ToolRuntime[ColourContext]) -> str:
    """Get the least favourite colour of the user"""
    return runtime.context.least_favourite_colour

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: gpt-5-nano es API de pago.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# agent = create_agent(
#     model="gpt-5-nano",
#     tools=[get_favourite_colour, get_least_favourite_colour],
#     context_schema=ColourContext
# )

In [7]:
# 🟢 ADAPTADO LOCAL
# Motivo: "gpt-5-nano" → get_model("gemma4:latest") (soporta tools). Ahora las tools leen el context vía ToolRuntime.
agent = create_agent(
    model=get_model("gemma4:latest"),
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColourContext
)

In [8]:
# Sin cambios
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='b3c292cf-1767-4283-8245-f36dd04a1a8d'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:latest', 'created_at': '2026-10-05T00:22:59.1203678Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2252187400, 'load_duration': 8195200, 'prompt_eval_count': 85, 'prompt_eval_duration': 1703127000, 'eval_count': 12, 'eval_duration': 536753000, 'logprobs': None, 'model_name': 'gemma4:latest', 'model_provider': 'ollama'}, id='lc_run--01a10971-359f-7892-a311-28fb7b22db34-0', tool_calls=[{'name': 'get_favourite_colour', 'args': {}, 'id': 'cfea5e7e-29f7-4576-a8f3-2a26209f17ee', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 85, 'output_tokens': 12, 'total_tokens': 97}),
              ToolMessage(content='blue', name='get_favourite_colour', id='927b24b2-1f12-4a21-95e5-649fc0e62635', tool_call_id='cfea5e7e-

In [9]:
# Sin cambios
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(favourite_colour="green")
)

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='379fcf6d-2376-4bf5-89b6-23950ebf66ce'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:latest', 'created_at': '2026-10-05T00:24:13.1812846Z', 'done': True, 'done_reason': 'stop', 'total_duration': 825518200, 'load_duration': 5637600, 'prompt_eval_count': 85, 'prompt_eval_duration': 282530000, 'eval_count': 12, 'eval_duration': 534955000, 'logprobs': None, 'model_name': 'gemma4:latest', 'model_provider': 'ollama'}, id='lc_run--01a10972-5c7a-79d1-8b08-cbbe4c4a60be-0', tool_calls=[{'name': 'get_favourite_colour', 'args': {}, 'id': 'c5e548e2-aa59-4a9f-85dd-71f4074b4597', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 85, 'output_tokens': 12, 'total_tokens': 97}),
              ToolMessage(content='green', name='get_favourite_colour', id='8efeec91-fa0a-4396-8a78-68e4e5271f1d', tool_call_id='c5e548e2-a

In [10]:
# 🟢 ADAPTADO LOCAL (celda nueva): ver qué recibe realmente el modelo.
# El context NO aparece en los mensajes: solo en el resultado de la tool (ToolMessage).
for m in response["messages"]:
    print(type(m).__name__, "→", m.tool_calls if getattr(m, "tool_calls", None) else m.content)

HumanMessage → What is my favourite colour?
AIMessage → [{'name': 'get_favourite_colour', 'args': {}, 'id': 'c5e548e2-aa59-4a9f-85dd-71f4074b4597', 'type': 'tool_call'}]
ToolMessage → green
AIMessage → The user's favourite colour is green.
